# UPenn-GBM: Predicting IDH1 Mutation Status from MRI Radiomics, and Its Association with Survival

**Data source:** [Imaging Data Commons (IDC)](https://imaging.datacommons.cancer.gov/) — `upenn_gbm` collection (UPenn-GBM, 630 patients).

**Why this dataset:** CTDC (the Clinical and Translational Data Commons) currently hosts no glioblastoma
patients and does not host imaging directly. UPenn-GBM in IDC has everything needed for this analysis:
structural MRI (with skull-stripped, co-registered `Processed_CaPTk` series), AI-generated tumor
sub-region segmentations (edema / enhancing tumor / necrosis), clinical `IDH1` mutation status, and
survival data (`survival_status`, `survival_from_surgery_days_updated`).

**Pipeline:**
1. Build a small demo cohort (all IDH-mutant patients + a matched sample of wildtype patients)
2. Identify the T1 post-contrast (`Processed_CaPTk`) series and its matching tumor segmentation for each patient
3. Download the DICOM image + segmentation
4. Extract PyRadiomics features from the whole-tumor mask
5. Train a classical ML model (logistic regression / random forest) to predict IDH1 status from radiomic features
6. Run a Kaplan-Meier survival analysis comparing IDH-mutant vs. IDH-wildtype patients (using their **actual**
   clinical labels, not model predictions)

**Caveats up front (read before drawing conclusions):**
- This is a **small demo cohort** (~35-40 patients out of 630), chosen for tractable download/runtime, not a
  publication-grade sample. IDH-mutant GBM is rare (only 19 of 630 patients here), so any classifier trained
  here is a proof-of-concept, not a validated model.
- The survival comparison uses the **full available clinical cohort** (not just the imaging demo subset) so
  the Kaplan-Meier curves reflect more patients than the radiomics model does.
- IDH-mutant glioma (and IDH-mutant GBM specifically) is well established in the literature as having
  better prognosis than IDH-wildtype — this notebook lets you check whether that holds in this cohort,
  it does not establish causality or generalize beyond UPenn-GBM.

**License:** UPenn-GBM imaging/segmentation data used here is **CC BY 4.0**. Cite using
`client.citations_from_selection(collection_id="upenn_gbm")` before publishing.

## 0. Setup

Install packages not already present. `idc-index`, `scikit-learn`, and `lifelines` were already available
in this environment; `pyradiomics`, `SimpleITK`, and `pydicom-seg` are needed for feature extraction from
the DICOM Segmentation objects.

In [ ]:
import importlib
import subprocess
import sys

required = {
    "idc_index": "idc-index",
    "SimpleITK": "SimpleITK",
    "pydicom": "pydicom",
    "pydicom_seg": "pydicom-seg",
    "radiomics": "pyradiomics",
    "lifelines": "lifelines",
    "sklearn": "scikit-learn",
}

missing = []
for module_name, pip_name in required.items():
    try:
        importlib.import_module(module_name)
    except ImportError:
        missing.append(pip_name)

if missing:
    print(f"Installing: {missing}")
    subprocess.run([sys.executable, "-m", "pip", "install", "--break-system-packages", *missing], check=True)
else:
    print("All required packages already installed.")

In [ ]:
import os
import glob

import numpy as np
import pandas as pd
import SimpleITK as sitk
import pydicom_seg
from radiomics import featureextractor

from idc_index import IDCClient

client = IDCClient()
print("IDC data version:", client.get_idc_version())

DATA_DIR = "./data/upenn_gbm_idh"
os.makedirs(DATA_DIR, exist_ok=True)

RANDOM_SEED = 42

## 1. Cohort selection

Load the UPenn-GBM clinical table and build:
- a **survival cohort** = every patient with a known IDH1 status (`Wildtype` or `Mutated`, excluding `NOS/NEC`)
- an **imaging demo cohort** = all 19 IDH-mutant patients + a random sample of ~20 IDH-wildtype patients,
  for radiomics extraction

In [ ]:
client.fetch_index("clinical_index")
clinical = client.get_clinical_table("upenn_gbm_clinical")

print(clinical.shape)
print(clinical["idh1"].value_counts(dropna=False))

# Survival cohort: drop ambiguous IDH calls
survival_cohort = clinical[clinical["idh1"].isin(["Wildtype", "Mutated"])].copy()
print(f"\nSurvival cohort: {len(survival_cohort)} patients")
print(survival_cohort["idh1"].value_counts())

In [ ]:
mutant = survival_cohort[survival_cohort["idh1"] == "Mutated"]
wildtype = survival_cohort[survival_cohort["idh1"] == "Wildtype"]

wildtype_sample = wildtype.sample(n=20, random_state=RANDOM_SEED)

imaging_cohort = pd.concat([mutant, wildtype_sample]).reset_index(drop=True)
print(f"Imaging demo cohort: {len(imaging_cohort)} patients "
      f"({len(mutant)} mutant, {len(wildtype_sample)} wildtype)")
imaging_cohort[["dicom_patient_id", "idh1", "survival_status", "survival_from_surgery_days_updated"]].head()

## 2. Identify imaging series

Not every patient has a usable T1 post-contrast (`Processed_CaPTk`) series with a matching AI tumor
segmentation (`Edema` / `Enhancing Lesion` / `Necrosis`) in IDC. Rather than sampling the imaging cohort
first and hoping for matches, we first find **all** patients in the survival cohort who have such a pair
(taking the earliest matching series per patient as the baseline scan), then build the imaging demo cohort
from that set.

In [ ]:
patient_ids = imaging_cohort["dicom_patient_id"].tolist()
patient_list_sql = ", ".join(f"'{p}'" for p in patient_ids)

client.fetch_index("seg_index")

series_pairs = client.sql_query(f"""
    WITH t1_post AS (
        SELECT PatientID, SeriesInstanceUID, StudyInstanceUID, SeriesDate, SeriesDescription,
               ROW_NUMBER() OVER (PARTITION BY PatientID ORDER BY SeriesDate ASC) AS rn
        FROM index
        WHERE collection_id = 'upenn_gbm'
          AND PatientID IN ({patient_list_sql})
          AND SeriesDescription = 't1 axial stealth-post : Processed_CaPTk'
    )
    SELECT t1.PatientID,
           t1.SeriesInstanceUID AS image_series,
           seg.SeriesInstanceUID AS seg_series,
           seg.SegmentedPropertyType_CodeMeanings,
           seg.total_segments
    FROM t1_post t1
    JOIN seg_index seg ON seg.segmented_SeriesInstanceUID = t1.SeriesInstanceUID
    WHERE t1.rn = 1
""")

print(f"Found {len(series_pairs)} image+segmentation pairs "
      f"(out of {len(patient_ids)} patients in the imaging cohort)")
series_pairs.head()

If the pair count is lower than the cohort size, some patients lack a baseline T1-post series or a matching
AI segmentation in IDC — those patients are dropped from the radiomics analysis (they remain in the survival
cohort).

## 3. Download DICOM image + segmentation series

This downloads ~35-40 image series and their matching segmentations. Each T1 series is a few MB to tens of
MB, so the full demo cohort should be on the order of a few hundred MB to ~1 GB.

In [ ]:
all_uids = list(series_pairs["image_series"]) + list(series_pairs["seg_series"])

client.download_from_selection(
    downloadDir=DATA_DIR,
    seriesInstanceUID=all_uids,
    dirTemplate="%PatientID/%Modality_%SeriesInstanceUID",
)

print("Download complete.")
print("Sample of downloaded folders:")
for p in sorted(glob.glob(os.path.join(DATA_DIR, "*", "*")))[:6]:
    print(" ", p)

## 4. Extract radiomic features

For each patient:
1. Load the T1 post-contrast volume with `SimpleITK`
2. Load the DICOM-SEG object with `pydicom_seg` and binarize it (`> 0`) to get a whole-tumor mask
   (combining edema + enhancing tumor + necrosis)
3. Run PyRadiomics with default settings to extract shape, first-order, and texture features

In [ ]:
extractor = featureextractor.RadiomicsFeatureExtractor()
# Whole-tumor mask is binary (0/1) -- treat label 1 as the region of interest
extractor.settings["label"] = 1


def load_image_series(series_dir: str) -> sitk.Image:
    reader = sitk.ImageSeriesReader()
    file_names = reader.GetGDCMSeriesFileNames(series_dir)
    reader.SetFileNames(file_names)
    return reader.Execute()


def load_seg_mask(series_dir: str) -> sitk.Image:
    seg_files = glob.glob(os.path.join(series_dir, "*.dcm"))
    dcm = pydicom_seg.MultiClassReader().read(
        __import__("pydicom").dcmread(seg_files[0])
    )
    seg_image = dcm.image  # SimpleITK image, integer-labeled
    binary_mask = sitk.BinaryThreshold(seg_image, lowerThreshold=1, upperThreshold=999, insideValue=1, outsideValue=0)
    return sitk.Cast(binary_mask, sitk.sitkUInt8)


def find_series_dir(patient_id: str, series_uid: str) -> str:
    matches = glob.glob(os.path.join(DATA_DIR, patient_id, f"*{series_uid}*"))
    if not matches:
        raise FileNotFoundError(f"No downloaded folder found for {patient_id} / {series_uid}")
    return matches[0]


records = []
for _, row in series_pairs.iterrows():
    patient_id = row["PatientID"]
    try:
        image_dir = find_series_dir(patient_id, row["image_series"])
        seg_dir = find_series_dir(patient_id, row["seg_series"])

        image = load_image_series(image_dir)
        mask = load_seg_mask(seg_dir)

        # Resample mask onto image grid if geometry differs slightly
        if mask.GetSize() != image.GetSize():
            mask = sitk.Resample(mask, image, sitk.Transform(), sitk.sitkNearestNeighbor, 0, mask.GetPixelID())

        features = extractor.execute(image, mask)
        feature_row = {
            k: v for k, v in features.items()
            if not k.startswith("diagnostics_")
        }
        feature_row["dicom_patient_id"] = patient_id
        records.append(feature_row)
        print(f"OK: {patient_id} ({len(feature_row) - 1} features)")
    except Exception as exc:
        print(f"SKIP: {patient_id} -- {exc}")

features_df = pd.DataFrame(records)
print(f"\nExtracted features for {len(features_df)} patients")
features_df.head()

## 5. Predict IDH1 status from radiomic features

Given the very small sample size (≤39 patients, only 19 mutant), we use **leave-one-out cross-validation
(LOOCV)** rather than a single train/test split, and report ROC-AUC across folds. This is a
proof-of-concept, not a validated diagnostic model -- treat the AUC as illustrative only.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, classification_report

model_df = features_df.merge(
    imaging_cohort[["dicom_patient_id", "idh1"]], on="dicom_patient_id"
)
y = (model_df["idh1"] == "Mutated").astype(int)
X = model_df.drop(columns=["dicom_patient_id", "idh1"])

print(f"Feature matrix: {X.shape}, positives (IDH-mutant): {y.sum()} / {len(y)}")

pipelines = {
    "LogisticRegression": Pipeline([
        ("scale", StandardScaler()),
        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced")),
    ]),
    "RandomForest": Pipeline([
        ("scale", StandardScaler()),
        ("clf", RandomForestClassifier(n_estimators=300, class_weight="balanced", random_state=RANDOM_SEED)),
    ]),
}

loo = LeaveOneOut()
for name, pipe in pipelines.items():
    proba = cross_val_predict(pipe, X, y, cv=loo, method="predict_proba")[:, 1]
    preds = (proba >= 0.5).astype(int)
    auc = roc_auc_score(y, proba)
    print(f"\n=== {name} (LOOCV) ===")
    print(f"ROC-AUC: {auc:.3f}")
    print(classification_report(y, preds, target_names=["Wildtype", "Mutated"], zero_division=0))

In [ ]:
# Inspect which radiomic features the random forest found most informative
rf = pipelines["RandomForest"]
rf.fit(X, y)
importances = pd.Series(rf.named_steps["clf"].feature_importances_, index=X.columns)
importances.sort_values(ascending=False).head(15)

## 6. Survival analysis: IDH-mutant vs. IDH-wildtype

Using the **full survival cohort** (all patients with known IDH1 status, not just the imaging demo subset)
and their **actual** clinical IDH1 labels:

- Time = `survival_from_surgery_days_updated`
- Event = 1 if `survival_status == "Deceased"` (or `"Deceased - uncertain date of death"`), else 0 (censored)

In [ ]:
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from lifelines.statistics import logrank_test

surv = survival_cohort.copy()
surv["time_days"] = pd.to_numeric(surv["survival_from_surgery_days_updated"], errors="coerce")
surv["event"] = surv["survival_status"].isin(["Deceased", "Deceased - uncertain date of death"]).astype(int)
surv = surv.dropna(subset=["time_days"])

print(f"Survival cohort with usable time data: {len(surv)} patients")
print(surv.groupby("idh1")["event"].agg(["count", "sum"]).rename(columns={"count": "n", "sum": "events"}))

kmf = KaplanMeierFitter()
fig, ax = plt.subplots(figsize=(7, 5))

for label, group in surv.groupby("idh1"):
    kmf.fit(group["time_days"], event_observed=group["event"], label=f"IDH1 {label} (n={len(group)})")
    kmf.plot_survival_function(ax=ax)

ax.set_title("UPenn-GBM: Overall Survival from Surgery by IDH1 Status")
ax.set_xlabel("Days from surgery")
ax.set_ylabel("Survival probability")
plt.tight_layout()
plt.show()

mut = surv[surv["idh1"] == "Mutated"]
wt = surv[surv["idh1"] == "Wildtype"]
result = logrank_test(mut["time_days"], wt["time_days"], mut["event"], wt["event"])
print(f"\nLog-rank test: statistic={result.test_statistic:.3f}, p-value={result.p_value:.4g}")

## 7. Citation

Generate proper attribution for the UPenn-GBM data used in this notebook before publishing any results.

In [ ]:
for citation in client.citations_from_selection(collection_id="upenn_gbm"):
    print(citation)

## Summary & next steps

- **Radiomics → IDH prediction**: the LOOCV AUC above is illustrative for a ~39-patient demo cohort with
  only 19 IDH-mutant cases. To do this properly: use the full 565-patient cohort, extract features from
  multiple sequences (T1-post, T2, FLAIR) and from each tumor sub-region separately (not just the combined
  whole-tumor mask), and validate on a held-out cohort (e.g. TCGA-GBM/TCGA-LGG, which also has IDH and
  imaging in IDC).
- **Survival**: the Kaplan-Meier comparison uses real clinical IDH1 labels for the full survival cohort.
  If the log-rank p-value is significant and the IDH-mutant curve sits above the wildtype curve, that's
  consistent with the well-established literature finding that IDH-mutant glioma has better prognosis.
- **Connecting back to radiomics**: a natural follow-on is to check whether the *radiomics-predicted*
  IDH status (rather than the true label) still stratifies survival -- i.e., whether an imaging-based
  IDH classifier could serve as a non-invasive prognostic proxy when molecular testing isn't available.